In [1]:
import os
os.getpid()

36641

In [2]:
# 导入APIKEY
from dotenv import find_dotenv, load_dotenv

_ = load_dotenv(find_dotenv(), override= True)

In [3]:
# 抓取文件夹路径
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/eric_zcz/Desktop/Eric_Project/agent")

folder_path = PROJECT_ROOT / "llm-universe" / "data_base" / "knowledge_db"
files_path = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        file_path = os.path.join(root, file)
        files_path.append(file_path)

In [4]:
# 加载每个文件（Langchain）
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import UnstructuredMarkdownLoader

loaders = []
for file_path in files_path:
    tag = file_path.split(".")[-1].lower()
    if (tag == 'pdf'):
        loaders.append(PyMuPDFLoader(file_path))
    if (tag == 'md'):
        loaders.append(UnstructuredMarkdownLoader(file_path))

texts = []
for loader in loaders:
    texts.extend(loader.load())


/var/folders/hv/jhxscpts3134d4_j91753j880000gn/T/ipykernel_36641/2396837798.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [5]:
# 切成chunks（Langchain）
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

split_text = text_splitter.split_documents(texts)

chunk_texts = [doc.page_content for doc in split_text]

In [5]:
# # 向量化（Langchain）
# from langchain_community.vectorstores import Chroma
# from langchain_openai import OpenAIEmbeddings
# import shutil

# embedding = OpenAIEmbeddings(
#     model='text-embedding-3-small'
# )

# persist_dir = "data_base/vector_db/chroma"

# if os.path.exists(persist_dir):
#     shutil.rmtree(persist_dir)

# vectordb = Chroma(
#     embedding_function=embedding,
#     persist_directory=persist_dir
# )

# batch_size = 50
# for i in range(0,  len(split_text), batch_size):
#     docs = split_text[i: i + batch_size]
#     vectordb.add_documents(docs)
        
# vectordb.persist()

# from langchain_chroma import Chroma
# from langchain_openai import OpenAIEmbeddings

# embedding = OpenAIEmbeddings(
#     model="text-embedding-3-small"
# )

# PROJECT_ROOT = Path(
#     subprocess.check_output(
#         ["git", "rev-parse", "--show-toplevel"],
#         text=True
#     ).strip()
# )

# persist_dir = PROJECT_ROOT / "Naive RAG" / "data_base" / "vector_db" / "chroma"

# vectordb = Chroma(
#     persist_directory=str(persist_dir),
#     embedding_function=embedding
# )



In [7]:
from pymilvus import MilvusClient

client = MilvusClient(uri="http://localhost:19530")
print(client.list_collections())

['RAG_collection']


In [8]:
from langchain_openai import OpenAIEmbeddings

embedding_fn = OpenAIEmbeddings(
    model="text-embedding-3-small"
    )

dimension = len(embedding_fn.embed_query("test"))
print(dimension)

vectors = embedding_fn.embed_documents(chunk_texts)



1536


In [8]:
if client.has_collection(collection_name="RAG_collection"):
    client.drop_collection(collection_name="RAG_collection")
client.create_collection(
    collection_name="RAG_collection",
    dimension=dimension,  # The vectors we will use in this demo has 768 dimensions
)

In [9]:
import time

data = [
    {"id": i, "vector": vectors[i], "text": chunk_texts[i], "subject": "agent"}
    for i in range(len(vectors))
]

batch_size = 100
for i in range(0, len(data), batch_size):
    batch = data[i:i + batch_size]
    client.insert(
        collection_name="RAG_collection",
        data=batch
    )
    print(f"Inserted {i + len(batch)} / {len(data)}")
    time.sleep(0.2)

Inserted 100 / 1082
Inserted 200 / 1082
Inserted 300 / 1082
Inserted 400 / 1082
Inserted 500 / 1082
Inserted 600 / 1082
Inserted 700 / 1082
Inserted 800 / 1082
Inserted 900 / 1082
Inserted 1000 / 1082
Inserted 1082 / 1082


In [9]:
from langchain_core.documents import Document

def vector_retriever(query: str, k: int = 10):
    query_vectors = embedding_fn.embed_query(query)
    
    res = client.search(
        collection_name="RAG_collection",  
        data=[query_vectors],
        limit=k,
        output_fields=["text", "subject"],
    )

    docs = []
    for result in res[0]:
        entity = result["entity"]
        docs.append(
            Document(
                page_content=entity.get("text", ""),
                metadata={
                    "subject": entity.get("subject", ""),
                    "retriever": "milvus",
                    "score": result.get("distance", None),
                }
            )
        )

    return docs

In [10]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(split_text)
bm25_retriever.k = 10

def bm25_search(query: str):
    
    docs = bm25_retriever.invoke(query)
    for doc in docs:
        doc.metadata["retriever"] = "bm25"

    return docs

In [11]:
def merge_and_deduplicate(milvus_docs, bm_docs):
    all_docs = milvus_docs + bm_docs
    seen = set()
    unique_docs = []
    for doc in all_docs:
        key = doc.page_content.strip()
        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)
    return unique_docs

In [12]:
from langchain_cohere import CohereRerank

def rerank(query: str, candidate_docs: list):
    reranker = CohereRerank(
        model="rerank-v3.5",
        top_n=8
    )

    reranked_docs = reranker.compress_documents(
        documents=candidate_docs,
        query=query
    )

    return reranked_docs

In [13]:
from langchain_core.runnables import RunnableLambda

def combine(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [14]:
def hybrid_retriever_with_rerank(query: str):

    vector_docs = vector_retriever(query)

    bm_docs = bm25_search(query)
    
    candidate_docs = merge_and_deduplicate(vector_docs, bm_docs)
    
    reranked_docs = rerank(query, candidate_docs)

    return reranked_docs

In [15]:
hybrid_retriever = RunnableLambda(
    hybrid_retriever_with_rerank
)

In [16]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.1",
    temperature=0
)

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch
from langchain_core.output_parsers import StrOutputParser


condense_question_system_template = (
    "请根据聊天记录和用户最新问题，"
    "把用户最新问题改写成一个可以独立理解的问题。"
    "不要回答问题，只需要返回改写后的问题。"

)

condense_question_prompt = ChatPromptTemplate.from_messages([
    ("system", condense_question_system_template),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

retriever_docs = RunnableBranch(
    (lambda x : not x.get("chat_history", False), RunnableLambda(lambda x : x["input"]) | hybrid_retriever),
    condense_question_prompt | llm | StrOutputParser() | hybrid_retriever
)


In [18]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

system_prompt = (
    "你是一个问答任务的助手。"
    "请使用检索到的上下文片段回答问题。"
    "如果你不知道答案就说不知道。"
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
])

qa_chain = (
    RunnablePassthrough.assign(
        context=lambda x: combine(x["context"])
    )
    |qa_prompt
    |llm
    |StrOutputParser()
)

qa_history_chain = (
    RunnablePassthrough.assign(
        context=retriever_docs
    ).assign(
        answer=qa_chain
    )
)

In [19]:
import os
from tavily import TavilyClient

tavily_client = TavilyClient()

In [20]:
def search_web(query: str) -> str:
    
    response = tavily_client.search(
        query=query,
        topic="general",
        search_depth="advanced",
        max_results=5,
        include_answer=False,
        include_raw_content=False,
    )

    results = response.get("results", [])
    if not results:
        return "No relevant web results found."

    blocks = []
    for i, item in enumerate(results, 1):
        title = item.get("title", "")
        url = item.get("url", "")
        content = item.get("content", "")
        score = item.get("score", "")

        blocks.append(
            f"[{i}]\n"
            f"title: {title}\n"
            f"url: {url}\n"
            f"content: {content}\n"
            f"score: {score}"
        )

    return "\n\n".join(blocks)

In [22]:
# res = qa_history_chain.invoke({
#     "input": "它的作者是谁？",
#     "chat_history": [
#         ("human", "西瓜书是什么？"),
#         ("ai", "西瓜书通常指周志华老师的《机器学习》一书。")
#     ]
# })

# print(res["answer"])

In [23]:
# res = qa_history_chain.invoke({
#     "input": "《机器学习》这本书的作者是谁？",
#     "chat_history": []
# })

# print(res["answer"])

In [21]:
def search_local_knowledge(query: str) -> str:

    result = qa_history_chain.invoke({
        "input": query,
        "chat_history": []
    })

    return result["answer"]

In [22]:
from openai import OpenAI
import json

openai_client = OpenAI()

In [23]:
tools = [
    {
        "type": "function",
        "name": "search_local_knowledge",
        "description": "Search the local machine learning knowledge base and return relevant evidence snippets.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "A machine learning question to search in the local knowledge base.",
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        },
        "strict": True,
    },
    {
        "type": "function",
        "name": "search_web",
            "description": "Search the web for recent or external information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "A query to search on the web."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            },
            "strict": True
    },
]

In [24]:
input_list = [
    {"role": "user", "content": "周志华《机器学习》这本书适不适合作为学习大模型 Agent 的基础入门？请结合书本知识和最近的公开资料分析。"},
]

response = openai_client.responses.create(
    model="gpt-5.5",
    tools=tools,
    input=input_list,
)

In [25]:
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "search_local_knowledge":
            query = json.loads(item.arguments)["query"]
            docs = search_local_knowledge(query)

            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": docs
            })
            
    if item.type == "function_call":
        if item.name == "search_web":
            query = json.loads(item.arguments)["query"]
            docs = search_web(query)

            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": docs
            })

In [29]:
response = openai_client.responses.create(
    model="gpt-5.5",
    instructions="Respond only with the docs generated by a tool.",
    tools=tools,
    input=input_list,
)

In [30]:
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final output:
{
  "id": "resp_016af8b0a2482d14006a2b6a72a6ac819c8758d63066b6a223",
  "created_at": 1781230194.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with the docs generated by a tool.",
  "metadata": {},
  "model": "gpt-5.5-2026-04-23",
  "object": "response",
  "output": [
    {
      "id": "rs_016af8b0a2482d14006a2b6a734114819c9c4fde96ea35cc1f",
      "summary": [],
      "type": "reasoning",
      "content": [],
      "encrypted_content": null,
      "status": null
    },
    {
      "id": "msg_016af8b0a2482d14006a2b6a7ac0ac819caa09abadf295965f",
      "content": [
        {
          "annotations": [],
          "text": "## 结论\n\n周志华《机器学习》（“西瓜书”）**适合作为学习大模型 Agent 的“机器学习基础补课书”**，但**不适合作为大模型 Agent 的直接入门教材**。\n\n更准确地说：\n\n- 如果你的目标是理解“什么是学习、泛化、过拟合、模型评估、监督/无监督/强化学习”等底层概念，西瓜书很合适。\n- 如果你的目标是马上学习 LLM Agent 的构建，例如 RAG、工具调用、函数调用、记忆、规划、多 Agent 协作、Agent 框架、评测与部署，那么西瓜书不够直接，需要搭配近年的大模型与 Agent 资料。\n\n---\n\n## 一、西瓜书能提供哪些有用基础？\n\n根据书本内容和相关知识整理，西瓜书主要覆盖的是经典机